# grad-expressed-in-out — worked example 2: Exponential backward: derivative equals the output itself

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `grad-expressed-in-out`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The exponential function has the remarkable property that its derivative equals itself: `d/dx exp(x) = exp(x) = out`. This is the clearest example of the 'grad expressed in out' pattern — once the forward pass stores `out = exp(x)`, the backward pass only needs `out` (no `exp` call, no access to `x`). By the chain rule, `dL/dx = grad_out * out`.

## Worked solution

**Step 1 — recall the math.** `d/dx exp(x) = exp(x)`. If we cached the forward result as `out`, then the local Jacobian is simply `out` itself.

**Step 2 — apply chain rule.** `dL/dx = grad_out * out`. This is one multiply — no transcendental function call at all during the backward pass.

**Step 3 — verify correctness.** We compare against `grad_out * t.exp(x)` (which recomputes exp). Both should match exactly.

**Step 4 — demonstrate what 'passing wrong x' does.** We call the backward function with a randomly wrong `x` and show it still gives the right answer (because `x` is unused), illustrating that `out` is the ONLY data needed.

In [ ]:
import torch as t

t.manual_seed(1)

def exp_back(grad_out: t.Tensor, out: t.Tensor, x: t.Tensor) -> t.Tensor:
    # d/dx exp(x) = exp(x) = out (the entire derivative IS the output)
    return grad_out * out

# Exercise
t.manual_seed(1)
x = t.randn(5)
out = t.exp(x)
grad_out = t.ones_like(x)

result = exp_back(grad_out, out, x)

# Verify: should equal grad_out * exp(x)
expected = grad_out * t.exp(x)
assert t.allclose(result, expected)
print(f"x:       {x.tolist()}")
print(f"out:     {out.tolist()}")
print(f"dL/dx:   {result.tolist()}")

# Demonstrate x-independence: wrong x, same out -> same answer
wrong_x = t.zeros_like(x)  # completely wrong x
result_wrong_x = exp_back(grad_out, out, wrong_x)
assert t.allclose(result_wrong_x, result), \
    "x should be unused: same out, different x must give same result"
print("Confirmed: exp_back is independent of x (uses only out).")